<a href="https://colab.research.google.com/github/Samir-atra/Barbados_Traffic_Analysis_Challenge_dev/blob/main/notebooks/mediapipe_video_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Mediapipe video analysis**
## Best run on GPU

# Imports

In [1]:
import os
import cv2
import subprocess
import pandas as pd
from google.cloud import storage
from collections import defaultdict
from IPython.display import Video, display
from google.colab.patches import cv2_imshow

# Set Up

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%%capture
!pip install mediapipe
!wget -q -O efficientdet.tflite -q https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/float16/1/efficientdet_lite0.tflite

In [4]:
!gcloud auth login # Authenticates your identity for general gcloud CLI command use e.g. gcloud storage cp, gcloud compute, gsutil
!gcloud auth application-default login # Authenticates your environment for API access e.g., storage.Client()

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=aAsyoqme7iRC9ye4V17FmqE9gbiaS3&prompt=consent&token_usage=remote&access_type=offline&code_challenge=yCgTMSdOtgUaHWP2ki6WHP72Fdwpc8yG-n4YUkMmI-g&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0ATX87lNMxPAbg-ApynUFrQAvZp96Tfkx-gLvuKkxKzOI0VTFZV1ySJ5UEWrnyGosXoRjZg

You are now logged in as [samiratra95@gmail.com].
Your current project 

In [5]:
client = storage.Client(project="brb-traffic")

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [6]:
base_dir = "/"
bucket_name = 'brb-traffic'
video_path='videos'
os.makedirs(video_path, exist_ok=True)

# Read Data

In [8]:
# Load the dataset
train = pd.read_csv(os.path.join(base_dir, '/content/Train.csv'))

# Extract camera ID and reconstruct the video path
def extract_camera_path(video):
    parts = video.split('/')
    filename = parts[-1]
    camera_id = filename.split('_')[0]
    return f"{camera_id}/{filename}"

train['videos'] = train['videos'].apply(extract_camera_path)

# Display shape and preview
display(train.shape, train.head())

(16076, 14)

,responseId,view_label,ID_enter,ID_exit,videos,video_time,datetimestamp_start,datetimestamp_end,date,signaling,congestion_enter_rating,congestion_exit_rating,time_segment_id,cycle_phase
0,zYkHaeOdB7XOnvgP3YW5kQs,Norman Niles #1,time_segment_0_Norman Niles #1_congestion_ente...,time_segment_0_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-00-45.mp4,2025-10-20 06:00:45,2025-10-20 06:00:45,2025-10-20 06:01:44,2025-10-20,none,free flowing,free flowing,0,train
1,NYsHaeCRLq-vnvgPjoXZqA0,Norman Niles #1,time_segment_1_Norman Niles #1_congestion_ente...,time_segment_1_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-01-45.mp4,2025-10-20 06:01:45,2025-10-20 06:01:45,2025-10-20 06:02:44,2025-10-20,none,free flowing,free flowing,1,train
2,A40HaYT8KNm7nvgPq8e12AU,Norman Niles #1,time_segment_2_Norman Niles #1_congestion_ente...,time_segment_2_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-02-45.mp4,2025-10-20 06:02:45,2025-10-20 06:02:45,2025-10-20 06:03:00,2025-10-20,none,free flowing,free flowing,2,train
3,EIsHaanDMK-vnvgPjoXZqA0,Norman Niles #1,time_segment_3_Norman Niles #1_congestion_ente...,time_segment_3_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-03-45.mp4,2025-10-20 06:03:45,2025-10-20 06:03:45,2025-10-20 06:04:44,2025-10-20,none,free flowing,free flowing,3,train
4,RYsHafSeMaqpmecP5vCV0AQ,Norman Niles #1,time_segment_4_Norman Niles #1_congestion_ente...,time_segment_4_Norman Niles #1_congestion_exit...,normanniles1/normanniles1_2025-10-20-06-04-45.mp4,2025-10-20 06:04:45,2025-10-20 06:04:45,2025-10-20 06:04:59,2025-10-20,none,free flowing,free flowing,4,train


In [9]:
ss = pd.read_csv(os.path.join(base_dir,'/content/SampleSubmission.csv'))
display(ss.shape,ss.head())

(880, 3)

,ID,Target,Target_Accuracy
0,time_segment_129_Norman Niles #1_congestion_en...,free flowing,free flowing
1,time_segment_130_Norman Niles #1_congestion_en...,heavy delay,heavy delay
2,time_segment_131_Norman Niles #1_congestion_en...,free flowing,free flowing
3,time_segment_132_Norman Niles #1_congestion_en...,heavy delay,heavy delay
4,time_segment_133_Norman Niles #1_congestion_en...,free flowing,free flowing


In [ ]:
# test = pd.read_csv(os.path.join(base_dir,'TestInputSegments.csv'))
# display(test.shape,test.head())

# Download files from a google cloud storage

In [15]:
blobs=train.videos.tolist()[200:300]
blobs

['normanniles1/normanniles1_2025-10-20-10-04-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-05-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-06-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-07-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-08-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-09-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-10-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-11-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-12-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-13-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-14-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-16-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-17-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-18-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-19-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-20-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-21-45.mp4',
 'normanniles1/normanniles1_2025-10-20-10-22-45.mp4',
 'normanniles1/normanniles1_

In [16]:
from google.api_core.exceptions import NotFound

for blob_name in blobs:
    blob = client.bucket(bucket_name).blob(blob_name)
    file_name = os.path.basename(blob_name)  # get last part after '/'
    local_path = os.path.join(video_path, file_name)

    print("Downloading:", blob_name)
    try:
        blob.download_to_filename(local_path)
        print("✅ Downloaded to:", local_path)
    except NotFound:
        print(f"❌ Error: {blob_name} not found in bucket {bucket_name}. Skipping.")
    except Exception as e:
        print(f"❌ An unexpected error occurred while downloading {blob_name}: {e}")

Downloading: normanniles1/normanniles1_2025-10-20-10-04-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-04-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-05-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-05-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-06-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-06-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-07-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-07-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-08-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-08-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-09-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-09-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-10-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-10-45.mp4
Downloading: normanniles1/normanniles1_2025-10-20-10-11-45.mp4
✅ Downloaded to: videos/normanniles1_2025-10-20-10-11-45.mp4
Download

# Delete videos

In [ ]:
import os
import shutil # Import shutil, though we'll adjust its usage

# Get the list of filenames that should exist in video_path
relevant_filenames = {os.path.basename(blob_path) for blob_path in blobs}

# Get the list of actual files in the video_path directory
current_video_files = os.listdir(video_path)

print(f"Number of relevant files (from blobs): {len(relevant_filenames)}")
print(f"Number of files currently in video_path: {len(current_video_files)}")

# Iterate through current files and delete those that ARE relevant
print("Deleting relevant videos from the directory...")
for filename in current_video_files:
    file_to_delete = os.path.join(video_path, filename)
    is_file = os.path.isfile(file_to_delete)
    is_mp4 = filename.lower().endswith('.mp4')

    print(f"Checking {filename}: Is relevant={filename in relevant_filenames}, Is file={is_file}, Is MP4={is_mp4}")

    if filename in relevant_filenames: # Condition inverted to delete relevant files
        if is_file and is_mp4:
            os.remove(file_to_delete)
            print(f"Deleted relevant MP4 file: {file_to_delete}")
    # else:
        # print(f"Keeping irrelevant file: {filename}") # Uncomment to see which files are kept

# Play some videos downloaded earlier

In [ ]:
def show_video(video_path, trim=False, duration=10, width=800):
    """
    Convert (and optionally trim) a video, suppress ffmpeg output, and display it inline.
    Parameters:
        video_path (str): Path to the input video file.
        trim (bool): Whether to trim the video to a short preview (default False).
        duration (int): Duration in seconds if trimming (default 10).
        width (int): Display width in pixels (default 800).
    """
    output_path = "preview.mp4"

    # Build ffmpeg command
    cmd = ["ffmpeg", "-i", video_path]
    if trim:
        cmd += ["-t", str(duration)]
    cmd += [output_path, "-y"]  # overwrite existing

    # Run ffmpeg silently (no stdout/stderr)
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Display the video inline
    display(Video(output_path, width=width, embed=True))

In [ ]:
counter = 0
for video in os.listdir(video_path):
    full_path = os.path.join(video_path, video)
    show_video(full_path)  # display the full video
    #show_video("normanniles1_2025-10-20-06-01-45.mp4", trim=True)     # 10s preview
    #show_video("normanniles1_2025-10-20-06-01-45.mp4", trim=True, duration=5)  # 5s preview
    counter += 1
    if counter == 2:
        break

# Mediapipe object detection setup

In [17]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import cv2
import os
from google.colab.patches import cv2_imshow
import pandas as pd # Import pandas for DataFrame creation
import re # Import regex for timestamp extraction


model_path = '/absolute/path/to/lite-model_efficientdet_lite0_detection_metadata_1.tflite'

# Populate congestion_map from the train DataFrame
congestion_map = {}
for index, row in train.iterrows():
    # The 'videos' column in train DataFrame contains paths like 'normanniles1/normanniles1_2025-10-20-10-04-45.mp4'
    # os.path.basename extracts 'normanniles1_2025-10-20-10-04-45.mp4'
    video_full_path = row['videos']
    video_filename_base = os.path.basename(video_full_path)
    congestion_map[video_filename_base] = {
        'congestion_enter_rating': row['congestion_enter_rating'],
        'congestion_exit_rating': row['congestion_exit_rating']
    }

BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path='/content/efficientdet.tflite', delegate=BaseOptions.Delegate.GPU),
    max_results=-1,
    score_threshold=0.5,
    running_mode=VisionRunningMode.VIDEO,
    category_allowlist= ["person", "truck", "car", "motorcycle", "bus"],
    )

# Initialize detector ONCE outside the loop
detector = ObjectDetector.create_from_options(options)

all_video_results = [] # List to store results for all videos
cumulative_timestamp_ms = 0 # Initialize a global cumulative timestamp

# Get list of video files and sort them by timestamp
def extract_timestamp_from_filename(filename):
    match = re.search(r'\d{4}-\d{2}-\d{2}-\d{2}-\d{2}-\d{2}', filename)
    if match:
        return match.group(0)
    return filename # Return original name if no timestamp found, for stable sorting

video_files_in_dir = [f for f in os.listdir(video_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
sorted_video_files = sorted(video_files_in_dir, key=extract_timestamp_from_filename)

try:
    for video_file_name in sorted_video_files:
        # Skip non-video files like .ipynb_checkpoints
        if not video_file_name.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
            print(f"Skipping non-video file: {video_file_name}")
            continue

        # Initialize objects dictionary for each video
        objects = {"car_left":0,"car_right":0,"truck_left":0,"truck_right":0,"motorcycle_left":0,"motorcycle_right":0,"person_left":0,"person_right":0,"bus_left":0,"bus_right":0}

        # Retrieve congestion ratings
        congestion_info = congestion_map.get(video_file_name, {'congestion_enter_rating': 'unknown', 'congestion_exit_rating': 'unknown'})
        enter_rating = congestion_info['congestion_enter_rating']
        exit_rating = congestion_info['congestion_exit_rating']

        full_path = os.path.join(video_path, video_file_name)
        cap = cv2.VideoCapture(full_path)
        video_file_fps = cap.get(cv2.CAP_PROP_FPS)
        frame_duration_ms = int(1000 / video_file_fps) if video_file_fps > 0 else 0 # Calculate duration of one frame for this video

        if not cap.isOpened():
            print(f"Error: Could not open video file {full_path}")
            continue

        print(f"Processing video: {video_file_name}")
        frame_index_within_video = 0 # Keeping this for local count if needed for other logic
        while True: # Loop until break
            ret, frame = cap.read()
            if ret:
                # Get frame dimensions
                frame_height, frame_width, _ = frame.shape
                frame_midpoint_x = frame_width / 2
            else: # Use else for consistency with if not ret
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

            # Use the global cumulative timestamp
            detection_result = detector.detect_for_video(mp_image, cumulative_timestamp_ms)

            # Increment the global cumulative timestamp for the next frame
            cumulative_timestamp_ms += frame_duration_ms

            for detection in detection_result.detections:
                bbox = detection.bounding_box

                category_name = detection.categories[0].category_name if detection.categories else "Unknown"
                score = round(detection.categories[0].score, 2) if detection.categories else 0.0

                # Calculate object's horizontal center for every detection
                object_center_x = bbox.origin_x + bbox.width / 2
                # Classify as 'left' or 'right' for every detection
                position_label = "left" if object_center_x < frame_midpoint_x else "right"

                if str(category_name) in["person", "truck", "car", "motorcycle", "bus"] and score >= 0.57:
                    # Update corresponding count only if criteria met
                    object_key = f'{category_name}_{position_label}'
                    if object_key in objects: # Ensure key exists before incrementing
                        objects[object_key] += 1

            # cv2_imshow(annotated_frame) # Display the annotated frame (commented out in original, keeping it that way)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
            frame_index_within_video += 1 # Increment local frame index
        print(f"Objects detected in {video_file_name}: {objects}")
        print(f"Congestion Enter Rating: {enter_rating}, Congestion Exit Rating: {exit_rating}")

        # Store results for the current video
        video_data = {
            'video_filename': video_file_name,
            'congestion_enter_rating': enter_rating,
            'congestion_exit_rating': exit_rating,
        }
        video_data.update(objects) # Add object counts to the dictionary
        all_video_results.append(video_data)

        cap.release() # Release the capture object after processing each video
        cv2.destroyAllWindows() # Close any OpenCV windows (important for local execution environments)
finally:
    # Close the detector after all videos are processed
    detector.close()

# Convert the list of dictionaries to a Pandas DataFrame
results_df = pd.DataFrame(all_video_results)
print("\n--- Results DataFrame ---")
display(results_df.head())


Processing video: normanniles1_2025-10-20-10-04-45.mp4
Objects detected in normanniles1_2025-10-20-10-04-45.mp4: {'car_left': 177, 'car_right': 225, 'truck_left': 26, 'truck_right': 37, 'motorcycle_left': 0, 'motorcycle_right': 0, 'person_left': 0, 'person_right': 0, 'bus_left': 0, 'bus_right': 3}
Congestion Enter Rating: light delay, Congestion Exit Rating: free flowing
Processing video: normanniles1_2025-10-20-10-05-45.mp4
Objects detected in normanniles1_2025-10-20-10-05-45.mp4: {'car_left': 17, 'car_right': 178, 'truck_left': 5, 'truck_right': 51, 'motorcycle_left': 0, 'motorcycle_right': 0, 'person_left': 0, 'person_right': 0, 'bus_left': 0, 'bus_right': 2}
Congestion Enter Rating: free flowing, Congestion Exit Rating: free flowing
Processing video: normanniles1_2025-10-20-10-06-45.mp4
Objects detected in normanniles1_2025-10-20-10-06-45.mp4: {'car_left': 163, 'car_right': 228, 'truck_left': 21, 'truck_right': 9, 'motorcycle_left': 0, 'motorcycle_right': 0, 'person_left': 0, 'pers

,video_filename,congestion_enter_rating,congestion_exit_rating,car_left,car_right,truck_left,truck_right,motorcycle_left,motorcycle_right,person_left,person_right,bus_left,bus_right
0,normanniles1_2025-10-20-10-04-45.mp4,light delay,free flowing,177,225,26,37,0,0,0,0,0,3
1,normanniles1_2025-10-20-10-05-45.mp4,free flowing,free flowing,17,178,5,51,0,0,0,0,0,2
2,normanniles1_2025-10-20-10-06-45.mp4,free flowing,free flowing,163,228,21,9,0,0,0,0,0,0
3,normanniles1_2025-10-20-10-07-45.mp4,free flowing,free flowing,82,262,9,17,0,0,0,0,0,13
4,normanniles1_2025-10-20-10-08-45.mp4,free flowing,free flowing,232,353,42,43,0,0,0,0,1,0


In [14]:
output_csv_path = 'video_analysis_results.csv'
results_df.to_csv(output_csv_path, index=False)
print(f"Results saved to {output_csv_path}")

Results saved to video_analysis_results.csv


# Objects counter

In [ ]:
"""
generate an initial counts sheet for the objects by setting ranges for the
number of objects and the classes in the training data and classify
the test data based on them.

check for  how to count the accurcy and f1-score
"""

# Objects dataset

In [ ]:
"""
video_list = []

for images in in the video:
    detect the most important 15 objects for each frame
    video_list.append(objects)

with open (video):
    create a csv file for the video with the objects name and score
"""

# Frame classification setup


In [ ]:
"""
build a Keras model to classify the frames into the required classes for the density of traffic
then find the most common to get the density of traffic for the full video.
"""